# SpurApp Packaging and Delivery Research

**Status:** draft  
**Date:** 2026-06-06  
**Beads epic:** `bd-3612`  
**Source spec:** `docs/superpowers/specs/2026-06-01-jute-app-notebook-as-application-container-design.ipynb`

This notebook captures the packaging and delivery design for **SpurApp**: the SPUR-branded notebook-as-application artifact. Jute remains the current local Tauri runtime shell; SpurApp is the shareable app package and application identity.

## Research Question

How should SpurApp package and deliver a notebook application so that:

- the `.ipynb` remains the source of truth,
- anywidget / AFM frontend cells are portable,
- Arrow port snapshots can be included without turning widget state into a bulk data channel,
- runtime dependencies are reproducible enough for recipients,
- delivery can scale from developer sharing to one-click desktop installation?

The answer should preserve the existing notebook-as-application architecture: Jute is the runtime shell, the notebook is the application manifest, the DAG engine is the cascade authority, and the Arrow port store is the durable data plane.

## Local Grounding

The existing Jute-App design already defines the key application primitives that SpurApp packages:

- the notebook id is the app/container identity,
- frontend cells declare UI bindings in notebook metadata,
- the reactive DAG engine remains central authority,
- control messages carry references and signals,
- bulk data stays in Arrow IPC files plus `manifest.json`,
- anywidget / AFM is the custom widget ABI.

Relevant local code paths:

- `crates/spur-notebook/jute-notebook/src-tauri/tauri.conf.json` already enables Tauri bundling and `.ipynb` file association.
- `xtask/src/main.rs::tauri_build_command` builds the Jute Tauri app bundle and injects staged resources such as the DuckDB extension.
- `crates/spur-core/src/notebook.rs::notebook_binary_path` discovers the bundled Jute app binary on macOS.
- `crates/spur-notebook/jute-notebook/src-tauri/src/kernel_provision.rs` provisions Python, Deno, Rust, and Go kernelspecs using local tools and the bundled `uv` sidecar.

Conclusion: the runtime bundle exists, but a notebook-specific portable SpurApp artifact does not.

## Reference Research

### anywidget / AFM

anywidget treats the frontend as an Anywidget Front-End Module (AFM): a web-standard ESM module that implements lifecycle hooks. Production projects bundle JS and CSS into static package assets, commonly through esbuild or Vite, then load those assets from the widget package. The Deno integration exists as `@anywidget/deno`, so Jute's Deno cell lane is a plausible authoring path.

Design consequence: inline Deno widgets are good for authoring and prototypes; production SpurApp packages should externalize AFM modules as content-hashed ESM/CSS assets.

### Jupyter Widgets Static Embedding

The classic Jupyter widget embed path serializes widget manager state and widget views into HTML using `application/vnd.jupyter.widget-state+json` and `application/vnd.jupyter.widget-view+json` script blocks. That is useful for static documents, but it does not preserve SpurApp's native runtime needs: the DAG engine, local bus, Arrow port store, and native polyglot kernels.

Design consequence: static embedding is an export tier, not the primary SpurApp package.

### JupyterLite

JupyterLite is optimized for static web deployment and browser-based kernels. It can host notebooks without a dedicated application server, but it cannot directly represent the local `ipc://` bus, native Tauri shell, and native Python/Deno/Rust/Go toolchain story.

Design consequence: JupyterLite-style export is useful only for a constrained web demo profile.

### Voila

Voila turns notebooks into server-hosted web applications. It is a strong reference for notebook-to-app delivery, but it moves the artifact into a server process and does not match the local notebook container model.

Design consequence: hosted SpurApp can borrow ideas from Voila later, but local capsule delivery should be the default.

### Tauri

Tauri provides platform installers and bundles: macOS app/DMG, Windows installers, and Linux packages. This is the correct runtime delivery channel for Jute itself, and a good optional wrapper for a specific SpurApp.

Design consequence: Tauri wraps the runtime and can wrap a capsule for one-click installation, but a signed per-notebook binary should not be the default sharing unit.

## Decision

Use a **`.spurapp` capsule** as the primary application artifact.

A `.spurapp` is a zip-like, content-addressed archive containing the notebook source, app manifest, bundled anywidget/AFM assets, dependency locks, optional port snapshots, and static resources.

Tauri remains the distribution path for the **Jute runtime**. A per-app Tauri installer is a secondary delivery tier built around the same `.spurapp` capsule.

This separates concerns cleanly:

- `.ipynb`: source-of-truth document,
- `.spurapp`: portable application capsule,
- `Jute.app` / Tauri installer: host runtime,
- optional per-app Tauri wrapper: one-click delivery for nontechnical recipients.

## Proposed Capsule Layout

```text
forecast-dashboard.spurapp/
  spur-app.json
  app.ipynb
  widgets/
    sha256-<hash>.mjs
    sha256-<hash>.css
  ports/
    manifest.json
    <port-name>@v<version>.arrow
  env/
    requirements.txt
    uv.lock
    deno.json
    deno.lock
    Cargo.lock
    go.sum
  resources/
    images/
    data/
  signatures/
    manifest.sig
```

`spur-app.json` is the capsule manifest. It should include:

```json
{
  "schema": "spur.app/v1",
  "name": "Forecast Dashboard",
  "entry_notebook": "app.ipynb",
  "open_mode": "app",
  "runtime": {
    "jute_min": "0.1.0",
    "features": ["frontend-cells", "anywidget-afm", "ports-arrow"]
  },
  "widgets": [
    {
      "module": "widgets/sha256-abc123.mjs",
      "css": "widgets/sha256-def456.css",
      "hash": "sha256:abc123"
    }
  ],
  "ports": {
    "include_snapshots": true,
    "manifest": "ports/manifest.json"
  },
  "dependencies": {
    "python": "env/uv.lock",
    "deno": "env/deno.lock",
    "rust": "env/Cargo.lock",
    "go": "env/go.sum"
  }
}
```

All paths are capsule-relative. Hashes make imports reproducible and allow Jute to cache widget assets across apps.

## anywidget / AFM Packaging Rules

1. **Development path:** allow inline Deno anywidget cells through `jsr:@anywidget/deno`.
2. **Production path:** export AFM as content-hashed ESM and CSS assets in `widgets/`.
3. **Notebook metadata:** frontend cells point at the asset hash or capsule-relative module path, not an unstable runtime `model_id`.
4. **Runtime state:** `model_id` remains runtime-only. Stable app state lives in notebook metadata and port names.
5. **Bulk data:** Arrow data never becomes durable synced widget state. Large payloads move through the port relay as refs or transferable buffers.
6. **Security:** AFM runs through the Jute host lifecycle and sandbox boundary; arbitrary kernel-authored HTML does not get same-origin Tauri access.

This follows the anywidget direction while preserving Jute's own DAG, ports, and security model.

## Runtime Dependency Model

The capsule should declare what it needs; Jute should decide whether to provision, reuse, or reject.

### Python

Use `uv.lock` when available. Include `requirements.txt` as a fallback for simple apps. Jute already has a bundled `uv` sidecar, so Python is the easiest dependency lane to make recipient-friendly.

### Deno / TypeScript

Include `deno.json` and `deno.lock`. Deno can resolve JSR and npm imports, but production capsules should prefer bundled AFM assets over live network fetches.

### Rust

Include `Cargo.lock` only for apps that use Rust cells. Rust kernels remain a more advanced delivery tier because recipients need the Rust toolchain or a future managed sidecar.

### Go

Include `go.mod` / `go.sum` only for apps that use Go cells. As with Rust, Go is best treated as an advanced or preflight-required lane until Jute can manage the toolchain.

### Preflight

Import should run a preflight report:

- required kernels,
- required sidecars/toolchains,
- package locks found or missing,
- port snapshots found or missing,
- AFM asset hashes verified or failed,
- unsafe/privileged actions requiring confirmation.

## Delivery Tiers

### Tier 1: Installed Jute + `.spurapp`

Best default for creators and technical recipients. Share the capsule; Jute imports it, verifies assets, provisions what it can, and opens the app in App mode.

### Tier 2: Installed Jute + raw `.ipynb`

Useful for author collaboration, but not reliable enough for final app delivery because widget assets, dependency locks, and port snapshots can be missing.

### Tier 3: Per-app Tauri wrapper

The same `.spurapp` is embedded as a resource in a signed Tauri app/installer. This is the right one-click channel for nontechnical users, demos, and customer delivery.

### Tier 4: Static web export

A constrained export for apps that can run in browser-only mode. This can borrow from JupyterLite and static widget embedding, but it is not equivalent to local SpurApp because native kernels and `ipc://` are absent.

### Tier 5: Hosted SpurApp

A future server-backed profile, closer to Voila. Out of scope for the first packaging implementation.

## Export / Import Flow

### Export

1. Validate notebook metadata and frontend cell declarations.
2. Resolve anywidget / AFM modules.
3. Bundle or copy AFM assets into `widgets/` with content hashes.
4. Collect dependency locks for used kernels.
5. Optionally include `ports/manifest.json` and selected Arrow snapshots.
6. Build `spur-app.json`.
7. Write `.spurapp` archive.

### Import

1. Verify `spur-app.json` schema and runtime compatibility.
2. Verify content hashes and signatures when present.
3. Unpack into Jute's app cache.
4. Register AFM assets in the widget module cache.
5. Run dependency preflight and provision supported kernels.
6. Hydrate port snapshots if included.
7. Open `app.ipynb` in App mode.

### Open Raw Notebook

Raw notebooks remain supported. If notebook metadata references missing capsule assets, Jute should surface a clear missing-asset diagnostic and offer to locate the capsule.

## Risks and Mitigations

| Risk | Mitigation |
|---|---|
| Capsule becomes a second source of truth | Keep `app.ipynb` as the only notebook source. Manifest records packaging metadata only. |
| Widget asset drift | Use content hashes and cache by hash. |
| Dependency lock mismatch | Preflight before running; do not silently install missing privileged toolchains. |
| Port snapshots become stale | Treat snapshots as last-known state; rerun DAG when sources or dependencies require it. |
| Large Arrow snapshots bloat packages | Make port snapshots opt-in and allow per-port include/exclude rules. |
| Security boundary weakens | Keep sandboxed output and AFM host API; no same-origin access for arbitrary output HTML. |
| Per-app binary maintenance overhead | Keep per-app Tauri wrapper optional and generated from the capsule. |

## Implementation Shape

Recommended first implementation plan:

1. Define `spur-app.json` schema and `.spurapp` archive reader/writer.
2. Add export command for current notebook to capsule.
3. Add import command that verifies, unpacks, and opens capsule entry notebook.
4. Add AFM asset resolver: inline, URL, local file, and content-hashed capsule asset.
5. Add dependency preflight report.
6. Add optional port snapshot inclusion.
7. Add one-click Tauri wrapper generation later.

Acceptance criteria for the first milestone:

- exporting a notebook produces a parseable `.spurapp`,
- importing the capsule opens the same notebook in Jute,
- content-hashed AFM assets are loaded from the capsule cache,
- missing dependencies are reported before execution,
- raw `.ipynb` sharing still works for simple notebooks.

## Sources

- Existing Jute-App design: `docs/superpowers/specs/2026-06-01-jute-app-notebook-as-application-container-design.ipynb`
- Local Tauri config: `crates/spur-notebook/jute-notebook/src-tauri/tauri.conf.json`
- Local Tauri build path: `xtask/src/main.rs::tauri_build_command`
- Local kernel provisioning: `crates/spur-notebook/jute-notebook/src-tauri/src/kernel_provision.rs`
- anywidget bundling: https://anywidget.dev/en/bundling/
- anywidget AFM: https://anywidget.dev/en/afm/
- `@anywidget/deno`: https://jsr.io/@anywidget/deno
- Jupyter widgets static embedding: https://ipywidgets.readthedocs.io/en/7.6.5/embedding.html
- JupyterLite deployment: https://jupyterlite.readthedocs.io/en/stable/
- Voila deployment: https://voila.readthedocs.io/en/latest/deploy.html
- Tauri distribution: https://v2.tauri.app/distribute/
- LabConstrictor notebook desktop app packaging reference: https://arxiv.org/abs/2603.10704